# 1. Environment Setup & Reproducibility
Install required packages, configure random seeds across all libraries for deterministic execution, and configure the compute device.

In [13]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


Configured device: cuda


# 2. Ontology Knowledge Graph Engine
Initializes the `OntologyEngine` to extract continuous 24-dimensional feature vectors representing emotions, cognitive appraisals, intensity, polarity, and negation cues.

In [14]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc_ekman.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Found ontology file at: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loading Knowledge Graph: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loaded 90420 triples.
Lexicon initialized with 2540 entries.


# 3. Lexical Coverage Analysis & Diagnostic Suite
Computes feature hit rates and active dimensions across datasets and verifies the engine against core linguistic diagnostic test cases.

In [15]:
# 1. Diagnostic test suite
test_cases = [
    ("tôi không hề thích nó", "Negation Scope Check"),
    ("cảm thấy không vui chút nào", "Flip Logic Check"),
    ("bún chả rất ngon", "Ambiguity Filter Check (Chả)"),
    ("làm hay nghỉ", "Ambiguity Filter Check (Hay)"),
    ("thất vọng tràn trề", "Multi-Word Expression Recognition"),
]

print("==================== ONTOLOGY DIAGNOSTIC REPORT ====================")
for text, desc in test_cases:
    vec, tokens, trace = ontengine.getvector(text, debug=True)
    matched = [t["phrase"] for t in trace] if trace else ["None"]
    is_neg = any(t["data"]["isnegation"] for t in trace) if trace else False

    labels = ontengine.ALLEMOTIONS
    emotions_vals = vec[:7]
    max_idx = np.argmax(emotions_vals)
    dominant = (
        f"{labels[max_idx]} ({emotions_vals[max_idx]:.4f})"
        if emotions_vals[max_idx] > 0
        else "Neutral/None"
    )

    print(f"\nScenario: {desc}")
    print(f"  - Input: '{text}'")
    print(f"  - Matched Phrases: {matched}")
    print(f"  - Negation Detected: {is_neg}")
    print(f"  - Dominant Emotion: {dominant}")
print("\n====================================================================")

==================== ONTOLOGY DIAGNOSTIC REPORT ====================

Scenario: Negation Scope Check
  - Input: 'tôi không hề thích nó'
  - Matched Phrases: ['không', 'thích']
  - Negation Detected: True
  - Dominant Emotion: Happiness (0.4510)

Scenario: Flip Logic Check
  - Input: 'cảm thấy không vui chút nào'
  - Matched Phrases: ['không', 'vui']
  - Negation Detected: True
  - Dominant Emotion: Happiness (0.4941)

Scenario: Ambiguity Filter Check (Chả)
  - Input: 'bún chả rất ngon'
  - Matched Phrases: ['rất']
  - Negation Detected: False
  - Dominant Emotion: Happiness (0.4189)

Scenario: Ambiguity Filter Check (Hay)
  - Input: 'làm hay nghỉ'
  - Matched Phrases: ['nghỉ']
  - Negation Detected: False
  - Dominant Emotion: Sadness (0.3888)

Scenario: Multi-Word Expression Recognition
  - Input: 'thất vọng tràn trề'
  - Matched Phrases: ['thất vọng']
  - Negation Detected: False
  - Dominant Emotion: Sadness (0.5764)



# 4. Dataset Loading, Preprocessing & DataLoaders
Loads the `vsfcekman-1` train, dev, and test sets, encodes the 7 Ekman emotion classes, extracts ontology features, and constructs PyTorch DataLoaders with balanced class weights.

In [16]:
# 1. Trích xuất vector cảm xúc thô VnEmoLex (Doãn & Lưu, 2022)
# Chỉ trích xuất và cộng dồn điểm cảm xúc của từ vựng theo từ điển, không dùng appraisal, polarity, negation rules hay similarity matrix
def get_vnemolex_vector(text, engine):
    tokens = ViTokenizer.tokenize(str(text).lower()).split()
    flat_tokens = [engine._norm(t) for t in tokens if engine._norm(t)]
    v_emo = np.zeros(len(engine.ALLEMOTIONS), dtype=np.float32)

    idx, n = 0, len(flat_tokens)
    while idx < n:
        matched_entry, matched_len = None, 0
        for l_span in range(min(engine.mwe_max_len, n - idx), 0, -1):
            phrase = " ".join(flat_tokens[idx : idx + l_span])
            if phrase in engine.lexicon_map:
                matched_entry = engine.lexicon_map[phrase]
                matched_len = l_span
                break

        if not matched_entry:
            idx += 1
            continue

        for ent in matched_entry:
            if ent.get("isnegation", False):
                continue
            score = ent.get("score", 1.0)
            for e, w in ent.get("emotions", {}).items():
                if e in engine.ALLEMOTIONS:
                    v_emo[engine.ALLEMOTIONS.index(e)] += score * w

        idx += matched_len

    return v_emo  # Vector cảm xúc thô 7 chiều


# 2. Khởi tạo Tokenizers cho cả 2 backbone
tokenizer_phobert = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")
tokenizer_visobert = AutoTokenizer.from_pretrained("uitnlp/visobert")


class DualBackboneDataset(Dataset):

    def __init__(self, df, tokenizer, engine, is_phobert=False, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.is_phobert = is_phobert
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.df.iloc[index]["Sentence"])
        label = self.df.iloc[index]["label_idx"]

        # PhoBERT bắt buộc tiền xử lý tách từ bằng pyvi trước khi mã hoá BPE
        input_text = ViTokenizer.tokenize(raw_text) if self.is_phobert else raw_text

        enc = self.tokenizer(
            input_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        vnemolex_vec = get_vnemolex_vector(raw_text, self.engine)
        ont_vector = self.engine.getvector(raw_text)
        if isinstance(ont_vector, tuple):
            ont_vector = ont_vector[0]

        return {
            "input_ids": enc["input_ids"].flatten(),
            "attention_mask": enc["attention_mask"].flatten(),
            "lexicon_features": torch.tensor(vnemolex_vec, dtype=torch.float),
            "ontology_features": torch.tensor(ont_vector, dtype=torch.float),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# 3. Load dữ liệu vsfcekman
BASE_DATA_PATH = "../../data/vsfc-ekman"

df_train = pd.read_csv(f"{BASE_DATA_PATH}/train.csv")
df_valid = pd.read_csv(f"{BASE_DATA_PATH}/dev.csv")
df_test = pd.read_csv(f"{BASE_DATA_PATH}/test.csv")

for df in [df_train, df_valid, df_test]:
    df.rename(
        columns={"Text": "Sentence", "Ekman_Label": "Emotion"}, inplace=True
    )
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

print(
    f"Loaded datasets: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}"
)

# Label Encoding (7 nhãn cảm xúc Ekman)
le = LabelEncoder()
all_labels = sorted(df_train["Emotion"].unique().astype(str))
le.fit(all_labels)
print(f"Classes ({len(le.classes_)}): {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df["label_idx"] = le.transform(df["Emotion"].astype(str))

# Dataloaders cho PhoBERT
BATCH_SIZE = 32
train_loader_pho = DataLoader(
    DualBackboneDataset(
        df_train, tokenizer_phobert, ontengine, is_phobert=True
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader_pho = DataLoader(
    DualBackboneDataset(
        df_valid, tokenizer_phobert, ontengine, is_phobert=True
    ),
    batch_size=BATCH_SIZE,
)
test_loader_pho = DataLoader(
    DualBackboneDataset(df_test, tokenizer_phobert, ontengine, is_phobert=True),
    batch_size=BATCH_SIZE,
)

# Dataloaders cho ViSoBERT
train_loader_viso = DataLoader(
    DualBackboneDataset(
        df_train, tokenizer_visobert, ontengine, is_phobert=False
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader_viso = DataLoader(
    DualBackboneDataset(
        df_valid, tokenizer_visobert, ontengine, is_phobert=False
    ),
    batch_size=BATCH_SIZE,
)
test_loader_viso = DataLoader(
    DualBackboneDataset(
        df_test, tokenizer_visobert, ontengine, is_phobert=False
    ),
    batch_size=BATCH_SIZE,
)

# Balanced Class Weights
train_labels = df_train["label_idx"].tolist()
class_weights = compute_class_weight(
    "balanced", classes=np.unique(train_labels), y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights: {weights_tensor.cpu().numpy()}")

Loaded datasets: Train=11426, Dev=1583, Test=3166
Classes (7): ['Anger' 'Disgust' 'Fear' 'Happiness' 'Neutral' 'Sadness' 'Surprise']
Class Weights: [ 0.76168257  2.6327188  41.85348     0.28987494  3.571741    0.65448505
 38.863945  ]


# 5. Model Architectures
Defines neural-symbolic architectures for 7-class emotion classification:
1. `ViSoBERT_Fusion`: Standard ViSoBERT backbone supporting baseline, concatenation, and gated projection.
2. `ViSoBERT_Residual_Fusion`: Multi-layer ontology projector with gated residual connections and LayerNorm.
3. `OntologyConceptProjector` & `ViSoBERT_Ontology_CrossAttention` (OCA): Token-level multi-head cross-attention over 5 projected ontology concept tokens.

In [17]:
# Import lexicon vs ontology comparative models from src.models
from src.models.lexicon_ontology_models import (
    Model_Baseline,
    Model_VnEmoLex_Concat,
    Model_Ontology_RawGate,
)

VNEMOLEX_DIM = 7
ONTOLOGY_DIM = 24
print('✅ Models loaded from src.models.lexicon_ontology_models: Model_Baseline, Model_VnEmoLex_Concat, Model_Ontology_RawGate')


# 6. Training Pipeline & Evaluation Utilities
Defines the training loop with validation early stopping and best checkpoint restoration.

In [18]:
def evaluate(model, loader, feature_key="lexicon_features"):
    """Compute accuracy and macro F1 score on a DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = batch[feature_key].to(device) if feature_key in batch else None

            outputs = model(ids, mask, feat)
            _, pr = torch.max(outputs, dim=1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(
    model,
    name,
    train_loader,
    valid_loader,
    feature_key="lexicon_features",
    epochs=15,
    patience=3,
):
    """Execute model training with weighted CrossEntropyLoss and early stopping."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=2e-5, weight_decay=0.01
    )
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = (
                batch[feature_key].to(device) if feature_key in batch else None
            )

            optimizer.zero_grad()
            outputs = model(ids, mask, feat)
            loss = criterion(outputs, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        val_acc, val_f1 = evaluate(model, valid_loader, feature_key=feature_key)
        avg_loss = total_loss / len(train_loader)
        print(
            f"Epoch {epoch + 1:02d}/{epochs:02d} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}",
            end=" ",
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            patience_counter = 0
            print("(New Best)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)

    return model

# 7. Model Training Across Configurations


In [19]:
NUM_CLASSES = len(le.classes_)
PHO_ID = "vinai/phobert-base-v2"
VISO_ID = "uitnlp/visobert"

# ==================== NHÓM 1: PHOBERT ====================
# 1. PhoBERT Baseline
m_pho_base = Model_Baseline(NUM_CLASSES, model_name=PHO_ID)
m_pho_base = train_experiment(
    m_pho_base,
    "1. PhoBERT Baseline",
    train_loader_pho,
    valid_loader_pho,
    feature_key="lexicon_features",
    patience=3,
)

# 2. PhoBERT + VnEmoLex Concat (Doãn & Lưu, 2022)
m_pho_lex = Model_VnEmoLex_Concat(NUM_CLASSES, model_name=PHO_ID)
m_pho_lex = train_experiment(
    m_pho_lex,
    "2. PhoBERT + VnEmoLex (Doãn & Lưu, 2022)",
    train_loader_pho,
    valid_loader_pho,
    feature_key="lexicon_features",
    patience=3,
)

# 3. PhoBERT + Full Ontology (RawGate)
m_pho_ont = Model_Ontology_RawGate(NUM_CLASSES, model_name=PHO_ID)
m_pho_ont = train_experiment(
    m_pho_ont,
    "3. PhoBERT + Full Ontology (RawGate)",
    train_loader_pho,
    valid_loader_pho,
    feature_key="ontology_features",
    patience=3,
)

# ==================== NHÓM 2: VISOBERT ====================
# 4. ViSoBERT Baseline
m_viso_base = Model_Baseline(NUM_CLASSES, model_name=VISO_ID)
m_viso_base = train_experiment(
    m_viso_base,
    "4. ViSoBERT Baseline",
    train_loader_viso,
    valid_loader_viso,
    feature_key="lexicon_features",
    patience=3,
)

# 5. ViSoBERT + VnEmoLex Concat (Doãn & Lưu, 2022)
m_viso_lex = Model_VnEmoLex_Concat(NUM_CLASSES, model_name=VISO_ID)
m_viso_lex = train_experiment(
    m_viso_lex,
    "5. ViSoBERT + VnEmoLex (Doãn & Lưu, 2022)",
    train_loader_viso,
    valid_loader_viso,
    feature_key="lexicon_features",
    patience=3,
)

# 6. ViSoBERT + Full Ontology (RawGate)
m_viso_ont = Model_Ontology_RawGate(NUM_CLASSES, model_name=VISO_ID)
m_viso_ont = train_experiment(
    m_viso_ont,
    "6. ViSoBERT + Full Ontology (RawGate)",
    train_loader_viso,
    valid_loader_viso,
    feature_key="ontology_features",
    patience=3,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. PhoBERT Baseline
Epoch 01/15 | Loss: 1.3959 | Val F1: 0.4523 | Val Acc: 0.7530 (New Best)
Epoch 02/15 | Loss: 1.0771 | Val F1: 0.4522 | Val Acc: 0.7555 (Patience: 1/3)
Epoch 03/15 | Loss: 0.8573 | Val F1: 0.4969 | Val Acc: 0.7802 (New Best)
Epoch 04/15 | Loss: 0.6615 | Val F1: 0.5600 | Val Acc: 0.7966 (New Best)
Epoch 05/15 | Loss: 0.4810 | Val F1: 0.5822 | Val Acc: 0.8143 (New Best)
Epoch 06/15 | Loss: 0.3488 | Val F1: 0.6234 | Val Acc: 0.8250 (New Best)
Epoch 07/15 | Loss: 0.2636 | Val F1: 0.6017 | Val Acc: 0.8307 (Patience: 1/3)
Epoch 08/15 | Loss: 0.2090 | Val F1: 0.6424 | Val Acc: 0.8301 (New Best)
Epoch 09/15 | Loss: 0.1622 | Val F1: 0.6249 | Val Acc: 0.8313 (Patience: 1/3)
Epoch 10/15 | Loss: 0.1462 | Val F1: 0.6389 | Val Acc: 0.8389 (Patience: 2/3)
Epoch 11/15 | Loss: 0.1173 | Val F1: 0.5924 | Val Acc: 0.8238 (Patience: 3/3)
Early stopping triggered after 11 epochs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 5. ViSoBERT + VnEmoLex (Doãn & Lưu, 2022)
Epoch 01/15 | Loss: 1.5357 | Val F1: 0.5117 | Val Acc: 0.7473 (New Best)
Epoch 02/15 | Loss: 1.0778 | Val F1: 0.5347 | Val Acc: 0.7461 (New Best)
Epoch 04/15 | Loss: 0.5102 | Val F1: 0.5470 | Val Acc: 0.7549 (Patience: 1/3)
Epoch 05/15 | Loss: 0.3267 | Val F1: 0.5775 | Val Acc: 0.7865 (New Best)
Epoch 06/15 | Loss: 0.2139 | Val F1: 0.5496 | Val Acc: 0.7928 (Patience: 1/3)
Epoch 07/15 | Loss: 0.1513 | Val F1: 0.5462 | Val Acc: 0.7757 (Patience: 2/3)
Epoch 08/15 | Loss: 0.1244 | Val F1: 0.5738 | Val Acc: 0.7865 (Patience: 3/3)
Early stopping triggered after 8 epochs.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 6. ViSoBERT + Full Ontology (RawGate)
Epoch 01/15 | Loss: 1.5298 | Val F1: 0.4747 | Val Acc: 0.6917 (New Best)
Epoch 02/15 | Loss: 1.0743 | Val F1: 0.5149 | Val Acc: 0.7461 (New Best)
Epoch 03/15 | Loss: 0.7542 | Val F1: 0.5176 | Val Acc: 0.7227 (New Best)
Epoch 04/15 | Loss: 0.4895 | Val F1: 0.5335 | Val Acc: 0.7808 (New Best)
Epoch 05/15 | Loss: 0.3131 | Val F1: 0.5341 | Val Acc: 0.7568 (New Best)
Epoch 06/15 | Loss: 0.2232 | Val F1: 0.5732 | Val Acc: 0.7934 (New Best)
Epoch 07/15 | Loss: 0.1417 | Val F1: 0.5483 | Val Acc: 0.7783 (Patience: 1/3)
Epoch 08/15 | Loss: 0.1144 | Val F1: 0.5478 | Val Acc: 0.7896 (Patience: 2/3)
Epoch 09/15 | Loss: 0.0911 | Val F1: 0.5369 | Val Acc: 0.7795 (Patience: 3/3)
Early stopping triggered after 9 epochs.


# 8. Final Benchmark Evaluation & Classification Reports
Collects model predictions on the test set and outputs macro F1, overall accuracy, and class-level classification metrics.

In [20]:
def get_predictions(model, loader, feature_key="lexicon_features"):
    """Extract ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            feat = (
                batch[feature_key].to(device) if feature_key in batch else None
            )

            outputs = model(ids, mask, feat)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# 1. Thu thập dự đoán PhoBERT
y_true, p_pho_base = get_predictions(
    m_pho_base, test_loader_pho, feature_key="lexicon_features"
)
_, p_pho_lex = get_predictions(
    m_pho_lex, test_loader_pho, feature_key="lexicon_features"
)
_, p_pho_ont = get_predictions(
    m_pho_ont, test_loader_pho, feature_key="ontology_features"
)

# 2. Thu thập dự đoán ViSoBERT
_, p_viso_base = get_predictions(
    m_viso_base, test_loader_viso, feature_key="lexicon_features"
)
_, p_viso_lex = get_predictions(
    m_viso_lex, test_loader_viso, feature_key="lexicon_features"
)
_, p_viso_ont = get_predictions(
    m_viso_ont, test_loader_viso, feature_key="ontology_features"
)

models_preds = [
    ("PhoBERT", "Baseline (No External)", p_pho_base),
    ("PhoBERT", "VnEmoLex Concat (Doãn & Lưu, 2022)", p_pho_lex),
    ("PhoBERT", "Full Ontology RawGate (Ours)", p_pho_ont),
    ("ViSoBERT", "Baseline (No External)", p_viso_base),
    ("ViSoBERT", "VnEmoLex Concat (Doãn & Lưu, 2022)", p_viso_lex),
    ("ViSoBERT", "Full Ontology RawGate (Ours)", p_viso_ont),
]

summary_records = []
for backbone, method, preds in models_preds:
    summary_records.append(
        {
            "Backbone": backbone,
            "Knowledge Integration": method,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n==================== TEST SET BENCHMARK RESULTS ====================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print(
    "\n========== REPORT: PhoBERT + VnEmoLex (Doãn & Lưu, 2022) =========="
)
print(classification_report(y_true, p_pho_lex, target_names=le.classes_, digits=4))

print(
    "\n========== REPORT: ViSoBERT + Full Ontology RawGate (Ours) =========="
)
print(classification_report(y_true, p_viso_ont, target_names=le.classes_, digits=4))


==================== TEST SET BENCHMARK RESULTS ====================
Backbone              Knowledge Integration  Accuracy  Macro F1
 PhoBERT             Baseline (No External)    0.8168    0.6472
 PhoBERT VnEmoLex Concat (Doãn & Lưu, 2022)    0.8301    0.6318
 PhoBERT       Full Ontology RawGate (Ours)    0.8193    0.6593
ViSoBERT             Baseline (No External)    0.7663    0.5704
ViSoBERT VnEmoLex Concat (Doãn & Lưu, 2022)    0.7748    0.5730
ViSoBERT       Full Ontology RawGate (Ours)    0.7767    0.5949

========== REPORT: PhoBERT + VnEmoLex (Doãn & Lưu, 2022) ==========
              precision    recall  f1-score   support

       Anger     0.7290    0.7378    0.7334       576
     Disgust     0.5075    0.4964    0.5018       137
        Fear     0.5000    0.3077    0.3810        13
   Happiness     0.9451    0.9653    0.9551      1586
     Neutral     0.7353    0.4491    0.5576       167
     Sadness     0.7301    0.7676    0.7484       680
    Surprise     0.7500    0.4286 

# 9. Neuro-Symbolic Rule-Based Post-Processing
Evaluates an override reasoning mechanism applying high-confidence ontology rules over uncertain baseline model outputs.

In [21]:
class NeuroSymbolicReasoner:
    """Combines neural probability outputs with symbolic ontology rules."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        le_classes,
        ont_threshold=0.6,
        model_conf_limit=0.9,
        is_phobert=False,
    ):
        self.model = (
            model.module if isinstance(model, nn.DataParallel) else model
        )
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.classes = le_classes
        self.cls_to_idx = {name: i for i, name in enumerate(le_classes)}
        self.ont_threshold = ont_threshold
        self.model_conf_limit = model_conf_limit
        self.is_phobert = is_phobert

    def infer(self, text):
        input_text = ViTokenizer.tokenize(text) if self.is_phobert else text
        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec = self.engine.getvector(text)
        if isinstance(vec, tuple):
            vec = vec[0]
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        pred_label_str = self.classes[pred_idx]

        final_idx = pred_idx

        # Rule: Nếu dự đoán Neutral nhưng độ tin cậy thấp, kiểm tra các cảm xúc mạnh từ ontology
        if (
            pred_label_str == "Neutral"
            and confidence < self.model_conf_limit
        ):
            ont_emotions = vec[:7]
            max_ont_idx = np.argmax(ont_emotions)
            max_ont_val = ont_emotions[max_ont_idx]
            max_ont_label = self.engine.ALLEMOTIONS[max_ont_idx]

            if max_ont_val > self.ont_threshold and max_ont_label != "Neutral":
                if max_ont_label in self.cls_to_idx:
                    final_idx = self.cls_to_idx[max_ont_label]

        return pred_idx, final_idx


def run_rule_evaluation(model, tokenizer, is_phobert=False, name="ViSoBERT"):
    reasoner = NeuroSymbolicReasoner(
        model=model,
        engine=ontengine,
        tokenizer=tokenizer,
        device=device,
        le_classes=le.classes_,
        ont_threshold=0.6,
        model_conf_limit=0.9,
        is_phobert=is_phobert,
    )

    y_base = []
    y_rule = []

    for txt in tqdm(df_test["Sentence"], desc=f"Rule Post-processing ({name})"):
        p_b, p_r = reasoner.infer(txt)
        y_base.append(p_b)
        y_rule.append(p_r)

    acc_base = accuracy_score(y_true, y_base)
    f1_base = f1_score(y_true, y_base, average="macro")
    acc_rule = accuracy_score(y_true, y_rule)
    f1_rule = f1_score(y_true, y_rule, average="macro")

    total_overrides = sum(1 for a, b in zip(y_base, y_rule) if a != b)
    print(
        f"[{name}] Total samples overridden: {total_overrides}/{len(df_test)} ({total_overrides / len(df_test) * 100:.2f}%)"
    )

    return {
        "Backbone": name,
        "Baseline Acc": acc_base,
        "Neuro-Symbolic Acc": acc_rule,
        "Delta Acc": acc_neuro - acc_base if 'acc_neuro' in locals() else acc_rule - acc_base,
        "Baseline F1": f1_base,
        "Neuro-Symbolic F1": f1_rule,
        "Delta F1": f1_rule - f1_base,
    }


records = []
records.append(
    run_rule_evaluation(
        m_pho_base, tokenizer_phobert, is_phobert=True, name="PhoBERT"
    )
)
records.append(
    run_rule_evaluation(
        m_viso_base, tokenizer_visobert, is_phobert=False, name="ViSoBERT"
    )
)

df_rule_res = pd.DataFrame(records)
print("\n============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============")
print(df_rule_res.round(4).to_string(index=False))

Rule Post-processing (PhoBERT):   0%|          | 0/3166 [00:00<?, ?it/s]

[PhoBERT] Total samples overridden: 1/3166 (0.03%)


Rule Post-processing (ViSoBERT):   0%|          | 0/3166 [00:00<?, ?it/s]

[ViSoBERT] Total samples overridden: 4/3166 (0.13%)

============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============
Backbone  Baseline Acc  Neuro-Symbolic Acc  Delta Acc  Baseline F1  Neuro-Symbolic F1  Delta F1
 PhoBERT        0.8168              0.8168        0.0       0.6472             0.6474    0.0002
ViSoBERT        0.7663              0.7663        0.0       0.5704             0.5679   -0.0025


# 10. Statistical Significance Testing (Paired Permutation Test)
Calculates exact permutation p-values over 10,000 resamples to evaluate statistical significance ($p < 0.05$) between model prediction vectors.

In [22]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    # PhoBERT
    (
        "PhoBERT: Baseline vs VnEmoLex (Doãn & Lưu)",
        p_pho_base,
        p_pho_lex,
    ),
    (
        "PhoBERT: VnEmoLex vs Full Ontology",
        p_pho_lex,
        p_pho_ont,
    ),
    (
        "PhoBERT: Baseline vs Full Ontology",
        p_pho_base,
        p_pho_ont,
    ),
    # ViSoBERT
    (
        "ViSoBERT: Baseline vs VnEmoLex (Doãn & Lưu)",
        p_viso_base,
        p_viso_lex,
    ),
    (
        "ViSoBERT: VnEmoLex vs Full Ontology",
        p_viso_lex,
        p_viso_ont,
    ),
    (
        "ViSoBERT: Baseline vs Full Ontology",
        p_viso_base,
        p_viso_ont,
    ),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
                                 Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)        Sig (F1)  p-val (Acc)              Sig (Acc)
 PhoBERT: Baseline vs VnEmoLex (Doãn & Lưu)        0.6472        0.6318     0.0154     0.63224 Not Significant      0.01580 Significant (p < 0.05)
         PhoBERT: VnEmoLex vs Full Ontology        0.6318        0.6593     0.0275     0.39486 Not Significant      0.05029        Not Significant
         PhoBERT: Baseline vs Full Ontology        0.6472        0.6593     0.0121     0.63964 Not Significant      0.70213        Not Significant
ViSoBERT: Baseline vs VnEmoLex (Doãn & Lưu)        0.5704        0.5730     0.0026     0.91631 Not Significant      0.15648        Not Significant
        ViSoBERT: VnEmoLex vs Full Ontology        0.5730        0.5949     0.0219     0.35096 Not Significant      0.79062        Not Significant
        ViSoBERT: Baseline vs Full O